# 05 — Self-Consistency Uncertainty

Self-consistency aggregates multiple sampled reasoning paths, using agreement as a confidence proxy.

## Agreement Across Samples as Confidence Proxy

**Self-consistency** (Wang et al., 2022) works for chain-of-thought reasoning:
1. Sample *k* diverse reasoning paths (temperature > 0)
2. Marginalise over the reasoning paths and take the majority answer
3. Use the fraction of paths agreeing with the majority as a confidence estimate

$$\text{confidence} = \frac{\text{count(majority answer)}}{k}$$

**Why it works**: the LLM often makes different errors across sampled paths, but correct reasoning paths tend to converge on the same answer. Majority vote improves accuracy by cancelling out idiosyncratic errors.

**Accuracy-confidence correlation**: self-consistency confidence correlates well with correctness across question types. For factual questions, agreement ≥ 0.8 strongly predicts correct answers.

**Cost**: requires *k* LLM calls (typically k=5-20). For expensive models, this is significant. Can be combined with semantic entropy for a richer picture: high agreement + low semantic entropy → high confidence.

**Limitations**: if a model systematically makes the same error, self-consistency won't help. Adversarial prompts or consistent misconceptions defeat it.

In [ ]:
# Self-consistency confidence + accuracy correlation
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

class ReasoningLLM:
    """
    Simulated LLM that samples multiple reasoning paths.
    Correctness probability follows question difficulty.
    """
    def __init__(self, base_accuracy=0.7):
        self.base_accuracy = base_accuracy

    def sample_path(self, question_id, path_id):
        """Returns: (final_answer, reasoning_text)"""
        np.random.seed(question_id * 100 + path_id)
        difficulty = np.random.uniform(0.4, 1.0)  # per-question
        # More likely to get the right answer for easier questions
        correct = np.random.random() < difficulty
        # Simulate a few possible answers
        if correct:
            return 'A', f'Reasoning path {path_id}: clear logic leads to A'
        else:
            # Wrong answer — but can be different wrong answers
            wrong = np.random.choice(['B', 'C', 'D'])
            return wrong, f'Reasoning path {path_id}: confused reasoning leads to {wrong}'

    def sample_k(self, question_id, k=10):
        """Sample k reasoning paths and return answers + consistency."""
        answers = [self.sample_path(question_id, i)[0] for i in range(k)]
        # Majority vote
        from collections import Counter
        vote = Counter(answers)
        majority_answer, majority_count = vote.most_common(1)[0]
        confidence = majority_count / k
        # Ground truth: most consistent answer across paths should be correct
        # (proxy: if majority is 'A' which is correct)
        is_correct = (majority_answer == 'A')
        return majority_answer, confidence, is_correct

llm = ReasoningLLM()

# Test 200 questions with k=10 samples each
results = []
for q_id in range(200):
    answer, conf, correct = llm.sample_k(q_id, k=10)
    results.append({'confidence': conf, 'correct': correct, 'answer': answer})

# Calibration analysis
n_bins = 5
edges = np.linspace(0, 1, n_bins + 1)
bin_data = []
for lo, hi in zip(edges[:-1], edges[1:]):
    mask = [(lo <= r['confidence'] < hi) for r in results]
    subset = [results[i] for i in range(len(results)) if mask[i]]
    if subset:
        avg_acc = np.mean([s['correct'] for s in subset])
        avg_conf = np.mean([s['confidence'] for s in subset])
        bin_data.append((avg_conf, avg_acc, len(subset)))

print('Self-consistency confidence vs accuracy:')
print(f'{"Confidence":>12} {"Accuracy":>10} {"Count":>8}')
for conf, acc, count in bin_data:
    print(f'{conf:>12.2f} {acc:>10.3f} {count:>8}')

# ECE
total = len(results)
ece = sum(len([r for r in results if b[0]-0.1 <= r['confidence'] < b[0]+0.1]) / total * abs(b[0]-b[1])
          for b in bin_data)
print(f'\nECE: {ece:.4f}')

In [ ]:
# Accuracy vs number of samples k
k_values = [1, 2, 3, 5, 7, 10, 15, 20]
accs_by_k = []

for k in k_values:
    correct_count = 0
    for q_id in range(200):
        answer, conf, correct = llm.sample_k(q_id, k=k)
        if correct:
            correct_count += 1
    accs_by_k.append(correct_count / 200)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Reliability
ax1.plot([0, 1], [0, 1], 'k--')
ax1.scatter([b[0] for b in bin_data], [b[1] for b in bin_data], color='steelblue', s=80)
ax1.set_xlabel('Confidence (fraction agreeing)')
ax1.set_ylabel('Accuracy')
ax1.set_title('Self-Consistency Reliability Diagram')

# Accuracy vs k
ax2.plot(k_values, accs_by_k, 'o-', color='steelblue')
ax2.set_xlabel('Number of samples k')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy vs Number of Samples (diminishing returns)')

plt.tight_layout()
plt.savefig('/tmp/self_consistency.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Accuracy improvement from k=1 to k=20: {accs_by_k[-1]-accs_by_k[0]:+.3f}')